# Ľudský zásah: Pred-akčné brány, rizikové úrovne a auditné denníky

README k tejto lekcii predstavuje Ľudský zásah krátkym úryvkom, ktorý požiada používateľa o `SCHVÁLENIE` alebo `ZAMITNUTIE` po tom, čo agent už vytvoril odpoveď. Tento vzor je dobrý východiskový bod, ale produkčné implementácie HITL obvykle potrebujú tri ďalšie prvky:

1. **pred-akčná brána**, ktorá sa spúšťa **predtým**, než agent vykoná riskantný krok, aby sa udržali pod kontrolou náklady, nezvratnosť a latencia.
2. **rizikové úrovne**, aby sa akcie s nízkym rizikom vykonávali automaticky, stredne rizikové akcie sa schvaľovali hromadne a len vysokorizikové akcie vyžadovali zásah človeka.
3. **auditný denník a slučka revízie**, aby každé rozhodnutie brány bolo zaznamenané ako JSONL, a zamietnutie vyzývalo agenta s štruktúrovaným dôvodom namiesto len vytlačenia `Revising...`.

Tento notebook na každom z nich stavia na rovnakých primitívach ako `06-system-message-framework.ipynb`. Beží kompletne v režime `DEMO_MODE = True` (nie je potrebný interaktívny vstup) alebo so skutočnými výzvami `input()`, keď je `DEMO_MODE = False`. Poznámka: v DEMO_MODE je opakovanie tretej cieľovej úlohy naprogramované, takže mechanika slučky je viditeľná celý priebeh. Skutočné revízne a znovu klasifikačné procesy vyžadujú `DEMO_MODE = False` a operátora.

**Mimo rozsahu (riešené v iných lekciách):** autentifikácia a kontrola prístupu (Lekcia 06 README hrozba 2), middleware volania nástrojov (Lekcia 14 MAF detailné spracovanie), viac-agentové diskusné vzory.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Vzor 1: Brána pred akciou

Úryvok HITL z README najskôr zavolá agenta, potom požiada používateľa o schválenie výstupu. Toto je **post-akčný** tok. Agent už vykonal svoju činnosť, takže náklady na volanie LLM sú už zaplatené a akýkoľvek vedľajší efekt (odoslaný e-mail, zapísaný riadok v databáze, zverejnený komentár) už nastal.

**Pre-akčný** tok vkladá bránu pred spustením rizikového kroku agentom. Agent navrhne akciu, brána rozhodne, či sa vykoná, a vedľajší efekt nastane iba po schválení.

| Aspekt | Schválenie po akcii (úryvok README) | Brána pred akciou (tento zápisník) |
|---|---|---|
| Kedy sa schválenie vykonáva? | Po tom, čo agent vygeneroval výstup | Pred vykonaním akéhokoľvek vedľajšieho efektu |
| Náklady na LLM pri zamietnutí | Už zaplatené | Zaplatené iba za návrh, nie za akciu |
| Nevratné vedľajšie efekty | Možné (akcia už prebehla) | Zamedzené |
| Jasnosť auditu | Schválenie je výpisová správa | Schválenie je JSON záznam s časovou značkou, akciou, dôvodom |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Vzor 2: Rizikové stupne

Nie každá akcia potrebuje ľudské schválenie. Vyhľadávanie len na čítanie prostredníctvom verejného API má iné riziká ako odoslanie e-mailu zákazníkovi. Rovnaké zaobchádzanie s oboma situáciami plytvá pozornosťou operátora a spomaľuje agenta.

Jednoduchý model s 3 stupňami:

| Stupeň | Príklady | Schvaľovací proces |
|---|---|---|
| `nízky` (len na čítanie) | Vyhľadávanie v databáze znalostí, hľadanie možností letu, načítanie verejnej webovej stránky | Automatické vykonanie, zaznamenané pre audit |
| `stredný` (lacnejšie zmeny) | Uloženie výsledku do cache, zvýšenie čítača, plánovanie pripomienky | Automatické vykonanie, ale s dennou hromadnou kontrolou |
| `vysoký` (externé alebo nezvratné) | Odoslanie emailu, zúčtovanie platby kartu, zverejnenie do verejného kanála | Zastavenie až po ľudskom schválení |

Toto je jedno rozdelenie na stupne. Produkčné systémy často používajú podrobnejšie stupne (napr. úrovne povolení AWS IAM, prístupové stupne na základe rolí). Verzia s 3 stupňami nižšie je najmenšia užitočná verzia pre agenta, ktorý mieša akcie len na čítanie a akcie so zásahom do stavu.

Klasifikátor nižšie používa heuristiku podľa kľúčových slov, aby zostalo demo deterministické a lacné. V produkčnom systéme by ste ho nahradili naučeným klasifikátorom alebo politikovým enginom.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Vzor 3: Auditný záznam a slučka revízie

`print("Response approved.")` nie je auditný záznam. Pre dôveru by malo byť každé rozhodnutie brány zaznamenané ako štruktúrovaná udalosť, ktorú neskôr môžete dotazovať, prehrávať alebo priložiť k prehľadu incidentu.

Dve súčasti:

1. **Iba dopĺňané JSONL.** Jeden riadok na rozhodnutie, s časovou značkou, akciou, úrovňou, rozhodnutím, dôvodom. Ľahké na grep, ľahké na odoslanie do skutočného úložiska záznamov neskôr.
2. **Slučka revízie pri zamietnutí.** Keď brána vráti `deny`, agent sa sám znovu vyzve s dôvodom zamietnutia v kontexte, aby sa ďalší návrh mohol vyhnúť problému.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Dodatočné zdroje

Niekoľko ďalších verejných projektov implementuje variácie týchto HITL (human-in-the-loop) vzorov. Porovnajte prístupy, aby ste našli ten, ktorý vyhovuje vášmu stacku:

- **LangChain** obaly nástrojov human-in-the-loop ([dokumentácia](https://python.langchain.com/docs/integrations/tools/human_tools)): obaly nástrojov, ktoré zastavia vykonávanie pre vstup od človeka.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ túto časť prerobil): používa rolu agenta špecificky na reprezentovanie človeka v multiagentných konverzáciách.
- **Microsoft Agent Framework (MAF)** middleware pre volanie funkcií ([dokumentácia](https://learn.microsoft.com/agent-framework/)): middleware, ktorý beží okolo každého volania nástroja/funkcie, vhodný pre riadiacu logiku a schvaľovacie procesy.

Každý projekt spracúva tri podvzory odlišne: LangChain ich obaluje ako nástroje, AutoGen používa rolu agenta a Microsoft Agent Framework používa middleware pre volanie funkcií. Prečítajte si jednu alebo dve implementácie dôkladne predtým, ako si vyberiete dizajn pre vlastného agenta.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
